# Easy GRPO Kaggle Run

This notebook runs the shaped/easy GRPO curriculum for OnCallEnv Red Shift. It is separate from the hard Qwen2.5 3B GRPO notebook so the high-score easy result stays clearly labeled.

## 1. GPU Check

Expected on Kaggle: Tesla T4 GPUs. If this does not show GPUs, the notebook is not attached to the Kaggle GPU kernel.

In [ ]:
!nvidia-smi

## 2. Bootstrap Repo

The Kaggle kernel cannot see the local laptop path. This clones or updates the pushed `round2-redshift` branch into `/kaggle/working`.

In [ ]:
import os
from pathlib import Path

Path('/kaggle/working').mkdir(parents=True, exist_ok=True)
os.chdir('/kaggle/working')
print('cwd:', os.getcwd())


In [ ]:
import os, shutil, subprocess, time
from pathlib import Path

REPO_URL = 'https://github.com/srimanreddy4/MetaHackathon-R2'
BRANCH = 'round2-redshift'
WORKDIR = Path('/kaggle/working/MetaHackathon-R2')

os.chdir('/kaggle/working')
if (WORKDIR / '.git').exists():
    os.chdir(WORKDIR)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    if WORKDIR.exists():
        backup = WORKDIR.with_name(f'{WORKDIR.name}.bak.{int(time.time())}')
        shutil.move(str(WORKDIR), str(backup))
        print('Moved non-git existing directory to', backup)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

print('cwd:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-5'], check=True)


In [ ]:
%cd /kaggle/working/MetaHackathon-R2
%env PYTHONPATH=src:scripts
!python - <<'PY'
import os
from pathlib import Path
print('cwd=', Path.cwd())
print('PYTHONPATH=', os.environ.get('PYTHONPATH'))
print('repo exists=', Path('src/oncallenv').exists())
PY


## 3. Install Dependencies

Run once per Kaggle session.

In [ ]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -r requirements-llm.txt


In [ ]:
# Fallback if dependency resolution fails:
# !python -m pip install -U transformers datasets accelerate trl peft bitsandbytes unsloth


## 4. Verify

Expected: `21 passed` and OpenEnv validation OK.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh verify

## 5. Easy Smoke Run

This validates easy prompts and shaped reward before spending more time. Expected score should be much higher than hard mode.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-smoke

In [ ]:
!cat training_results/unsloth_grpo_qwen3b_easy_smoke/summary.json

## 6. Easy Main Run

This is the high-score shaped-curriculum run: easy prompts, dense partial-credit reward, 300 steps.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-main

In [ ]:
!cat training_results/unsloth_grpo_qwen3b_easy/summary.json
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-summary

## 7. Optional Fallback

Use this only if the 3B easy run OOMs or behaves badly.

In [ ]:
# MODEL_NAME=unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit bash scripts/run_kaggle_qwen3b_grpo.sh easy-main

## 8. Archive Outputs

Download `/kaggle/working/qwen3b_grpo_results.tar.gz` from Kaggle outputs.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh archive